# 10 — Cryptographic Implementation Security

## What This Is
Using correct cryptographic algorithms is necessary but not sufficient — implementations must also be secure. Common implementation pitfalls: nonce reuse in AES-GCM, RSA padding oracle, timing leakage in comparison, weak random number generators, and protocol-level vulnerabilities (CBC padding oracle, BEAST, POODLE).

## Why It Matters
The Sony PS3 was broken because ECDSA reused nonces (allowing key recovery from 2 signatures). AES-GCM with nonce reuse leaks the authentication key and decrypts all past messages. WEP failed entirely due to RC4 IV reuse. Heartbleed leaked OpenSSL's memory due to a missing bounds check.

## Where the Community Stands
Nacl/libsodium and BoringSSL avoid footguns through opinionated APIs. NIST SP 800-57 guides key management. RFC 8446 (TLS 1.3) eliminated every known protocol vulnerability in older TLS versions.

## Key Principles
- Never roll your own crypto
- Use authenticated encryption (AES-GCM, ChaCha20-Poly1305)
- Unique nonce per message — catastrophic if reused
- Constant-time comparison for secrets
- CSPRNG for all randomness

In [ ]:
import hashlib
import hmac
import secrets
import struct
import os
from dataclasses import dataclass
from typing import List, Tuple, Dict

# --- Nonce reuse attack on AES-GCM (conceptual) ---
# Real AES-GCM requires PyCryptodome; we simulate the key recovery concept

def simulate_gcm_nonce_reuse_vuln():
    """
    If nonce N is reused for two messages M1 and M2:
    C1 = M1 XOR keystream(N)
    C2 = M2 XOR keystream(N)
    => C1 XOR C2 = M1 XOR M2  (keystream cancels!)
    If attacker knows M1, they recover M2.
    Authentication key H also leaks via GCM polynomial.
    """
    keystream = secrets.token_bytes(32)  # simulated AES-CTR keystream

    M1 = b'GET /admin?token=supersecret1234'
    M2 = b'GET /api/users?data=confidential'

    C1 = bytes(a ^ b for a, b in zip(M1, keystream[:len(M1)]))
    C2 = bytes(a ^ b for a, b in zip(M2, keystream[:len(M2)]))

    # Attacker observes C1, C2 and knows M1
    recovered_M2 = bytes(a ^ b ^ c for a, b, c in zip(C1, C2, M1))

    return {
        'M1':           M1.decode(),
        'M2':           M2.decode(),
        'recovered_M2': recovered_M2.decode('ascii', 'ignore'),
        'success':      recovered_M2 == M2,
    }

result = simulate_gcm_nonce_reuse_vuln()
print('AES-GCM Nonce Reuse Attack:')
print(f'  M1 (known):     {result["M1"]}')
print(f'  M2 (secret):    {result["M2"]}')
print(f'  Recovered M2:   {result["recovered_M2"]}')
print(f'  Full recovery:  {result["success"]}')
print()
print('Fix: generate a fresh random 96-bit nonce per message')
print('     or use XChaCha20-Poly1305 (extended 192-bit nonce)')

In [ ]:
# --- Common implementation vulnerabilities ---

# 1. Weak RNG
import random as insecure_rng

def generate_token_insecure(length: int = 32) -> str:
    """INSECURE: uses Python random — seeded with system time, predictable."""
    charset = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    return ''.join(insecure_rng.choice(charset) for _ in range(length))

def generate_token_secure(length: int = 32) -> str:
    """SECURE: secrets module uses OS CSPRNG."""
    charset = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    return ''.join(secrets.choice(charset) for _ in range(length))

# 2. ECB mode (penguin problem)
def ecb_determinism_demo():
    """ECB encrypts identical blocks identically — patterns visible."""
    # Simulate by showing identical input blocks produce identical output
    key = secrets.token_bytes(16)
    plaintext_blocks = [
        b'AAAAAAAAAAAAAAAA',  # block 1
        b'BBBBBBBBBBBBBBBB',  # block 2
        b'AAAAAAAAAAAAAAAA',  # block 3 — same as block 1!
    ]
    # Simulate ECB: each block independently encrypted with same key
    def fake_ecb_encrypt(block: bytes, key: bytes) -> bytes:
        return hashlib.sha256(key + block).digest()[:16]  # deterministic, not real AES

    ecb_out = [fake_ecb_encrypt(b, key) for b in plaintext_blocks]
    return {
        'block1_ct': ecb_out[0].hex()[:16],
        'block2_ct': ecb_out[1].hex()[:16],
        'block3_ct': ecb_out[2].hex()[:16],
        'blocks_1_and_3_identical': ecb_out[0] == ecb_out[2],
    }

ecb = ecb_determinism_demo()
print('ECB mode determinism (penguin problem):')
print(f'  Block 1 ciphertext: {ecb["block1_ct"]}...')
print(f'  Block 2 ciphertext: {ecb["block2_ct"]}...')
print(f'  Block 3 ciphertext: {ecb["block3_ct"]}...')
print(f'  Blocks 1&3 identical: {ecb["blocks_1_and_3_identical"]} <- pattern visible!')
print('  Fix: use AES-GCM or AES-CBC with random IV')

## TLS Version Vulnerability History
Each TLS version removed specific protocol vulnerabilities. TLS 1.3 eliminated all of them by design: no RSA key exchange (forward secrecy mandatory), no CBC mode, no renegotiation, no compression.

In [ ]:
TLS_VULN_HISTORY = {
    'SSL 2.0 (1995)': {
        'deprecated': '2011 (RFC 6176)',
        'vulnerabilities': [
            'DROWN (decryption from 2048 SSLv2 handshakes)',
            'No message authentication',
            'Weak export ciphers (40-bit)',
        ]
    },
    'SSL 3.0 (1996)': {
        'deprecated': '2015 (RFC 7568)',
        'vulnerabilities': ['POODLE (CBC padding oracle)', 'No AEAD support']
    },
    'TLS 1.0 (1999)': {
        'deprecated': '2021 (RFC 8996)',
        'vulnerabilities': ['BEAST (CBC IV predictability)', 'CRIME (compression oracle)', 'RC4 statistical bias']
    },
    'TLS 1.1 (2006)': {
        'deprecated': '2021 (RFC 8996)',
        'vulnerabilities': ['LUCKY13 (CBC timing)', 'No AEAD requirement', 'RC4 still supported']
    },
    'TLS 1.2 (2008)': {
        'deprecated': 'Not deprecated — still widely used',
        'vulnerabilities': [
            'ROBOT (RSA PKCS#1 v1.5 padding oracle if misconfigured)',
            'SWEET32 (64-bit block cipher birthday attack)',
            'Optional forward secrecy (DHE/ECDHE not mandatory)',
        ]
    },
    'TLS 1.3 (2018)': {
        'deprecated': 'N/A — current standard',
        'vulnerabilities': [
            'None known at protocol level',
            'Removed: RSA key exchange, CBC mode, compression, renegotiation',
            'Mandatory forward secrecy (ECDHE only)',
            '0-RTT: replay risk for non-idempotent requests',
        ]
    },
}

for version, info in TLS_VULN_HISTORY.items():
    print(f'\n{version}')
    print(f'  Status: {info["deprecated"]}')
    for v in info['vulnerabilities']:
        print(f'  - {v}')

In [ ]:
# Cryptographic algorithm selection guide
CRYPTO_RECOMMENDATIONS = {
    'Symmetric encryption': {
        'RECOMMENDED': ['AES-256-GCM', 'ChaCha20-Poly1305'],
        'ACCEPTABLE':  ['AES-256-CBC (with HMAC)', 'AES-256-CTR (with HMAC)'],
        'AVOID':       ['AES-ECB', 'DES', '3DES', 'RC4', 'Blowfish'],
    },
    'Asymmetric encryption': {
        'RECOMMENDED': ['ECDH P-256/P-384', 'X25519 (Curve25519)'],
        'ACCEPTABLE':  ['RSA-4096 (OAEP padding)'],
        'AVOID':       ['RSA < 2048', 'RSA PKCS#1 v1.5', 'DSA', 'DH < 2048'],
    },
    'Digital signatures': {
        'RECOMMENDED': ['Ed25519', 'ECDSA P-256/P-384'],
        'ACCEPTABLE':  ['RSA-PSS 4096'],
        'AVOID':       ['RSA PKCS#1 v1.5 signatures', 'DSA', 'ECDSA without deterministic k'],
    },
    'Hashing': {
        'RECOMMENDED': ['SHA-256', 'SHA-384', 'SHA-512', 'SHA3-256', 'BLAKE3'],
        'ACCEPTABLE':  ['SHA-224'],
        'AVOID':       ['MD5', 'SHA-1', 'CRC32'],
    },
    'Password hashing': {
        'RECOMMENDED': ['Argon2id', 'scrypt'],
        'ACCEPTABLE':  ['bcrypt (cost >= 12)'],
        'AVOID':       ['MD5', 'SHA-256 (without KDF)', 'SHA-1', 'PBKDF2-SHA1'],
    },
    'Key derivation': {
        'RECOMMENDED': ['HKDF-SHA256', 'Argon2id'],
        'ACCEPTABLE':  ['PBKDF2-SHA256 (>= 600000 rounds)'],
        'AVOID':       ['PBKDF2-SHA1', 'MD5-based KDFs', 'Simple concatenation'],
    },
}

print('Cryptographic Algorithm Selection Guide:')
for category, recs in CRYPTO_RECOMMENDATIONS.items():
    print(f'\n{category}:')
    for status, algs in recs.items():
        print(f'  {status}: {", ".join(algs)}')